# Entraînement YOLOv8m sur le dataset Spot

Notebook d'entraînement du détecteur YOLOv8m sur les frames DARPA annotées
automatiquement par Grounding DINO puis validées manuellement.

Il suppose que les scripts du dossier `scripts/` ont déjà été exécutés :

1. `02_download_darpa_videos.sh` — téléchargement des vidéos DARPA
2. `03_extract_frames.py` — extraction des frames à 1 FPS
3. `04_annotate_dino.py` — annotation automatique par Grounding DINO
4. `05_verify_annotations.py` — validation manuelle des annotations

**Prérequis** : GPU CUDA recommandé

**Hyperparamètres** : 50 epochs, image size 640, batch size 16, modèle de
base `yolov8m.pt`

In [ ]:
# Vérification de l'environnement GPU
import subprocess

try:
    output = subprocess.check_output(
        ["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
        text=True,
    )
    print("GPU détecté :")
    print(output)
except (subprocess.CalledProcessError, FileNotFoundError):
    print("Aucun GPU CUDA détecté. L'entraînement sera très lent sur CPU.")

In [ ]:
# Installation d'Ultralytics (YOLOv8) si nécessaire
import importlib.util

if importlib.util.find_spec("ultralytics") is None:
    %pip install -q ultralytics
else:
    print("Ultralytics déjà installé.")

## Configuration du chemin

Le notebook s'exécute depuis `notebooks/`. On remonte d'un cran pour pointer
sur la racine du projet, ce qui rend les chemins relatifs (`configs/...`,
`data/...`) cohérents avec le reste du repo.

In [ ]:
import os
from pathlib import Path

# Remonter à la racine du projet (parent du dossier notebooks/)
project_root = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
os.chdir(project_root)

print(f"Racine projet : {Path.cwd()}")
assert (Path("configs/dataset.yaml")).exists(), "configs/dataset.yaml introuvable"
assert (Path("data/annotations_dino")).exists(), "data/annotations_dino introuvable"

## Entraînement

Lancement du `train` avec les hyperparamètres validés en mai 2026.
Les résultats (poids, courbes, matrices de confusion) sont écrits dans
`runs/detect/train/` à la racine du projet.

In [ ]:
from ultralytics import YOLO

model = YOLO("yolov8m.pt")  # initialisation depuis les poids COCO

results = model.train(
    data="configs/dataset.yaml",
    epochs=50,
    imgsz=640,
    batch=16,
    name="spot_yolov8m",
    project="runs/detect",
)

## Évaluation sur le split test

Le `val` sur le split test fournit les métriques finales du modèle :
précision, rappel, mAP50, mAP50-95, et la matrice de confusion sauvegardée
dans le dossier d'évaluation.

In [ ]:
metrics = model.val(data="configs/dataset.yaml", split="test")

print(f"mAP50      : {metrics.box.map50:.4f}")
print(f"mAP50-95   : {metrics.box.map:.4f}")
print(f"Précision  : {metrics.box.mp:.4f}")
print(f"Rappel     : {metrics.box.mr:.4f}")

## Visualisation de prédictions

Quelques détections sur le split test, pour une inspection qualitative.

In [ ]:
import random
import matplotlib.pyplot as plt
from PIL import Image

test_images = list(Path("data/annotations_dino/test/images").glob("*.jpg"))
samples = random.sample(test_images, min(6, len(test_images)))

predictions = model.predict(source=[str(p) for p in samples], save=False)

fig, axes = plt.subplots(2, 3, figsize=(15, 10))
for ax, pred in zip(axes.flat, predictions):
    annotated = pred.plot()  # BGR numpy array
    ax.imshow(Image.fromarray(annotated[..., ::-1]))  # BGR -> RGB
    ax.set_title(Path(pred.path).name, fontsize=9)
    ax.axis("off")

plt.tight_layout()
plt.show()

## Export du modèle

Le checkpoint final (`best.pt`) est copié dans `models/` à la racine du
projet, prêt à être uploadé sur Hugging Face Hub.

Voir `models/README.md` pour les instructions d'upload.

In [ ]:
import shutil

source = Path("runs/detect/spot_yolov8m/weights/best.pt")
destination = Path("models/best.pt")

destination.parent.mkdir(exist_ok=True)
shutil.copy(source, destination)

size_mb = destination.stat().st_size / (1024 * 1024)
print(f"Modèle exporté : {destination} ({size_mb:.1f} MB)")